# Feeding the CPU data into a GPU

### Importing packages

In [3]:
import pandas as pd
import numpy as np

### Simulating raw data

In [4]:
print("Generating synthetic tick data...")
n_ticks = 500_000 # Half a million ticks

prices = 1.1000 + np.cumsum(np.random.normal(0, 0.0001, n_ticks))
volumes = np.random.randint(10_000, 500_000, n_ticks) # Volume per tick
timestamps = pd.date_range("2019-01-01", periods=n_ticks, freq="1S")

tick_df = pd.DataFrame({'price': prices, 'volume': volumes}, index=timestamps)
print(tick_df)

Generating synthetic tick data...
                        price  volume
2019-01-01 00:00:00  1.100162  478206
2019-01-01 00:00:01  1.099967  314029
2019-01-01 00:00:02  1.100119  261852
2019-01-01 00:00:03  1.100324  306065
2019-01-01 00:00:04  1.100401  412098
...                       ...     ...
2019-01-06 18:53:15  1.138347  440014
2019-01-06 18:53:16  1.138455  141782
2019-01-06 18:53:17  1.138590  124424
2019-01-06 18:53:18  1.138621  122836
2019-01-06 18:53:19  1.138621  323632

[500000 rows x 2 columns]


C:\Users\ofurn\AppData\Local\Temp\ipykernel_2540\1795031084.py:6: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  timestamps = pd.date_range("2019-01-01", periods=n_ticks, freq="1S")


### Compress to volume bars

In [ ]:
def create_volume_bars(df, volume_threshold=5_000_000):
    print(f"compressing ticks into {volume_threshold:,} volume bars...")

    cumulative_vol = df['volume'].cumsum()

    bar_ids = cumulative_vol // volume_threshold

    bars = df.groupby(bar_ids).agg(
            timestamp=('price', lambda x: x.index[-1]), # End time of the bar
            open=('price', 'first'),
            high=('price', 'max'),
            low=('price', 'min'),
            close=('price', 'last'),
            volume=('volume', 'sum')
        )

    bars.set_index('timestamp', inplace=True)

    bars['returns'] = bars['close'].pct_change().fillna(0)

volume_bars = create_volume_bars(tick_df)
print(f"reduced {len(tick_df)} ticks to {len(volume_bars)} volume bars")